# Check robustness of ability stratification with IRT(Item Response Theory)

## Imports

In [ ]:
!pip install -q py-irt

import os
import json
import pandas as pd
import numpy as np

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = (
    "/content/drive/MyDrive/education-ml-research/"
    "ASSISTments2009"
)

DATA_PATH = os.path.join(
    DATA_DIR,
    "skill_builder_data.csv"
)

QUARTILE_PATH = os.path.join(
    DATA_DIR,
    "student_ability_quartiles.csv"
)

IRT_INPUT_PATH = os.path.join(
    DATA_DIR,
    "assistments_irt.jsonlines"
)

IRT_OUTPUT_DIR = os.path.join(
    DATA_DIR,
    "irt_1pl_output"
)

os.makedirs(IRT_OUTPUT_DIR, exist_ok=True)

print(DATA_PATH)
print(QUARTILE_PATH)
print(IRT_OUTPUT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/student_ability_quartiles.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2009/irt_1pl_output


## Load dataset

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Original shape:", df.shape)
print("Students:", df["user_id"].nunique())
print("Skills:", df["skill_id"].nunique())

/tmp/ipykernel_945/2644713000.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, encoding="latin1")


Original shape: (525534, 30)
Students: 4217
Skills: 123


## Prepare IRT responses

In [ ]:
irt_df = df[
    ["user_id", "skill_id", "correct"]
].copy()

# Remove interactions without a skill
irt_df = irt_df.dropna(
    subset=["user_id", "skill_id"]
)

# Convert response to numeric
irt_df["correct"] = pd.to_numeric(
    irt_df["correct"],
    errors="coerce"
)

# Keep only binary responses
irt_df = irt_df[
    irt_df["correct"].isin([0, 1])
].copy()

# Reset index
irt_df = irt_df.reset_index(drop=True)

print("Valid interactions:", len(irt_df))
print("Students:", irt_df["user_id"].nunique())
print("Skills:", irt_df["skill_id"].nunique())

Valid interactions: 459208
Students: 4163
Skills: 123


## Convert IDs to contiguous responses

In [ ]:
student_ids = np.sort(
    irt_df["user_id"].unique()
)

skill_ids = np.sort(
    irt_df["skill_id"].unique()
)

student_to_idx = {
    student_id: i
    for i, student_id in enumerate(student_ids)
}

skill_to_idx = {
    skill_id: i
    for i, skill_id in enumerate(skill_ids)
}

irt_df["model"] = irt_df["user_id"].map(
    student_to_idx
)

irt_df["item"] = irt_df["skill_id"].map(
    skill_to_idx
)

print("Number of students:", len(student_ids))
print("Number of skills:", len(skill_ids))

print(irt_df.head())

Number of students: 4163
Number of skills: 123
   user_id  skill_id  correct  model  item
0    64525       1.0        1      7     0
1    64525       1.0        1      7     0
2    70363       1.0        0     15     0
3    70363       1.0        1     15     0
4    70363       1.0        0     15     0


## Create PyTorch tensors

In [ ]:
import torch

models = torch.tensor(
    irt_df["model"].values,
    dtype=torch.long
)

items = torch.tensor(
    irt_df["item"].values,
    dtype=torch.long
)

responses = torch.tensor(
    irt_df["correct"].values,
    dtype=torch.float32
)

print("Models:", models.shape)
print("Items:", items.shape)
print("Responses:", responses.shape)

print("Response mean:", responses.mean().item())

Models: torch.Size([459208])
Items: torch.Size([459208])
Responses: torch.Size([459208])
Response mean: 0.690373420715332


## Check data

In [ ]:
print(
    "Model index range:",
    models.min().item(),
    "to",
    models.max().item()
)

print(
    "Item index range:",
    items.min().item(),
    "to",
    items.max().item()
)

print(
    "Response values:",
    torch.unique(responses)
)

Model index range: 0 to 4162
Item index range: 0 to 122
Response values: tensor([0., 1.])


## Fit IRT model

In [ ]:
import pyro

device = "cpu"

print("Device:", device)

models = models.to(device)
items = items.to(device)
responses = responses.to(device)

pyro.clear_param_store()

irt_model = OneParamLog(
    priors="hierarchical",
    device=device,
    num_items=len(skill_ids),
    num_models=len(student_ids),
    verbose=True
)

NUM_EPOCHS = 1000

irt_model.fit(
    models=models,
    items=items,
    responses=responses,
    num_epochs=NUM_EPOCHS
)

Device: cpu
[epoch 0001] loss: 31845653.3727
[epoch 0101] loss: 263937.1411
[epoch 0201] loss: 252933.7523
[epoch 0301] loss: 277663.6866
[epoch 0401] loss: 295822.9432
[epoch 0501] loss: 269285.2817
[epoch 0601] loss: 260792.3232
[epoch 0701] loss: 265615.1519
[epoch 0801] loss: 280587.3444
[epoch 0901] loss: 270449.7520
[epoch 1000] loss: 262040.1620


## Get theta

In [ ]:
param_store = pyro.get_param_store()

print("Available parameters:")
for name in param_store.keys():
    print(name)

## Theta is relative ranking of student ability, a higher value corresponds to higher ability
theta = (
    param_store["loc_ability"]
    .detach()
    .cpu()
    .numpy()
)

print("Theta shape:", theta.shape)
print("Number of theta estimates:", len(theta))

print("\nTheta statistics:")
print(pd.Series(theta).describe())

irt_scores = pd.DataFrame({
    "user_id": student_ids,
    "theta": theta
})

print(irt_scores.head())

print("NaN theta:", irt_scores["theta"].isna().sum())
print(
    "Infinite theta:",
    np.isinf(irt_scores["theta"]).sum()
)

Available parameters:
loc_mu_b
scale_mu_b
loc_mu_theta
scale_mu_theta
alpha_b
beta_b
alpha_theta
beta_theta
loc_ability
scale_ability
loc_diff
scale_diff
Theta shape: (4163,)
Number of theta estimates: 4163

Theta statistics:
count    4163.000000
mean        0.591335
std         1.912386
min        -7.407269
25%        -0.322326
50%         0.616842
75%         1.438337
max         7.536068
dtype: float64
   user_id     theta
0       14 -0.671458
1    21825  1.066383
2    51950  1.914564
3    52613 -0.207613
4    53167  0.588570
NaN theta: 0
Infinite theta: 0


## Merge ability quartiles with IRT scores

In [ ]:
accuracy_ability = pd.read_csv(
    QUARTILE_PATH
)

print(accuracy_ability.head())
print(accuracy_ability.columns)
print("Students:", len(accuracy_ability))

comparison = accuracy_ability.merge(
    irt_scores,
    on="user_id",
    how="inner"
)

print("Matched students:", len(comparison))
print(comparison.head())

   user_id  total_correct  total_attempts  accuracy ability_quartile
0       14             13              51  0.254902               Q1
1    21825             21              29  0.724138               Q3
2    51933              0               1  0.000000               Q1
3    51950              5               6  0.833333               Q4
4    52613              4               7  0.571429               Q2
Index(['user_id', 'total_correct', 'total_attempts', 'accuracy',
       'ability_quartile'],
      dtype='object')
Students: 4217
Matched students: 4163
   user_id  total_correct  total_attempts  accuracy ability_quartile     theta
0       14             13              51  0.254902               Q1 -0.671458
1    21825             21              29  0.724138               Q3  1.066383
2    51950              5               6  0.833333               Q4  1.914564
3    52613              4               7  0.571429               Q2 -0.207613
4    53167            242             

## Check correlation

In [ ]:
from scipy.stats import pearsonr, spearmanr

## Pearson correlation measures the linear correlation between accuracy and IRT theta
pearson_r, pearson_p = pearsonr(
    comparison["accuracy"],
    comparison["theta"]
)

## High Spearman correlation means that students ranked highly by raw accuracy are also ranked highly by IRT ability.
spearman_r, spearman_p = spearmanr(
    comparison["accuracy"],
    comparison["theta"]
)

print(f"Pearson correlation: {pearson_r:.4f}")
print(f"Pearson p-value: {pearson_p:.4e}")

print(f"\nSpearman correlation: {spearman_r:.4f}")
print(f"Spearman p-value: {spearman_p:.4e}")

Pearson correlation: 0.9016
Pearson p-value: 0.0000e+00

Spearman correlation: 0.9390
Spearman p-value: 0.0000e+00


## IRT quartile check

In [ ]:
comparison["irt_quartile"] = pd.qcut(
    comparison["theta"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print(
    comparison[
        [
            "user_id",
            "accuracy",
            "ability_quartile",
            "theta",
            "irt_quartile"
        ]
    ].head(20)
)

comparison["quartile_match"] = (
    comparison["ability_quartile"]
    == comparison["irt_quartile"]
)

agreement_rate = comparison["quartile_match"].mean()

print(f"Exact quartile agreement: {agreement_rate:.2%}")

quartile_table = pd.crosstab(
    comparison["ability_quartile"],
    comparison["irt_quartile"],
    rownames=["Accuracy Quartile"],
    colnames=["IRT Quartile"]
)

print(quartile_table)

## In the table, most students were classified as the same quartile by both accuracy and IRT theta, with main discrepancies being between neighboring quartiles.
## This means IRT quartile and accuracy quartile are highly correlated.

    user_id  accuracy ability_quartile     theta irt_quartile
0        14  0.254902               Q1 -0.671458           Q1
1     21825  0.724138               Q3  1.066383           Q3
2     51950  0.833333               Q4  1.914564           Q4
3     52613  0.571429               Q2 -0.207613           Q2
4     53167  0.691429               Q3  0.588570           Q2
5     54318  0.833333               Q4  1.397604           Q3
6     58161  0.400000               Q1 -0.636804           Q1
7     64525  0.838750               Q4  1.694596           Q4
8     64531  1.000000               Q4  5.224028           Q4
9     64532  1.000000               Q4  5.219016           Q4
10    64535  0.816327               Q4  0.871075           Q3
11    64550  1.000000               Q4  5.514315           Q4
12    64634  0.589744               Q2 -0.058193           Q2
13    69931  0.500000               Q2 -0.190141           Q2
14    70190  0.545455               Q2  0.270790           Q2
15    70

## Conclusion

The IRT robustness check showed that the accuracy-based measure of student ability was highly consistent with an independent 1-parameter IRT measure that accounts for differences in item difficulty. Across the 4,163 students with both measures, overall accuracy and IRT theta had a strong positive correlation (Pearson's *r* = 0.902; Spearman's *ρ* = 0.939). Both correlations had p-values that rounded to 0.0000, meaning that, assuming there were no true relationship between accuracy and IRT ability, the probability of observing correlations this strong would be extremely small. This provides strong statistical evidence that the two measures are positively associated. Additionally, 74.68% of students were assigned to the same ability quartile using both methods, with most remaining differences occurring between adjacent quartiles. Overall, these results indicate that the accuracy-based ability stratification is robust to using an alternative IRT-based measure of ability, supporting its use in the subsequent reliability analysis.
